# Trabalhando com arquivos grandes

**Para quem já se sentiu confortável com os outros notebooks.** Aqui estão as
três técnicas que permitem analisar bases de milhões de linhas sem travar o
computador — e sem precisar de um computador potente.

**Tempo estimado:** 5 a 8 minutos.

## Preparação

In [2]:
%pip install pysus==2.10.6 nest_asyncio duckdb -q
import nest_asyncio
nest_asyncio.apply()
print("Ambiente pronto.")

Note: you may need to restart the kernel to use updated packages.
Ambiente pronto.


## O problema

A dengue de 2024 tem 6,5 milhões de notificações e 121 colunas. Pedir tudo de
uma vez consome cerca de **29 GB de memória** — mais do que o Google Colab
oferece. O notebook trava.

As três saídas, da mais simples à mais poderosa:

## Técnica 1 — Pegar o caminho em vez da tabela

Com `as_dataframe=False`, a biblioteca baixa o arquivo e devolve **onde ele
está**, sem carregar nada na memória.

In [3]:
from pysus import sinan

caminhos = sinan(disease="DENG", year=2024, as_dataframe=False)
arquivo = caminhos[0]

import os
print("Arquivo:", arquivo)
print(f"Tamanho em disco: {os.path.getsize(arquivo) / 1e6:.0f} MB")

Arquivo: C:\Users\Alexandre\pysus\downloads\ducklake\sinan\DENGBR24.parquet
Tamanho em disco: 151 MB


## Técnica 2 — Ler apenas as colunas necessárias

O formato Parquet guarda os dados por coluna, então é possível ler três colunas
sem tocar nas outras 118.

In [4]:
import pandas as pd
import time

inicio = time.time()
dados = pd.read_parquet(arquivo, columns=["DT_NOTIFIC", "SG_UF_NOT", "CLASSI_FIN"])
segundos = time.time() - inicio

print(f"{len(dados):,} linhas em {segundos:.1f}s")
print(f"Memória: {dados.memory_usage(deep=True).sum() / 1e9:.2f} GB "
      f"(contra ~29 GB com todas as colunas)")

6,564,924 linhas em 0.4s


Memória: 1.06 GB (contra ~29 GB com todas as colunas)


## Técnica 3 — Consultar sem carregar (SQL com duckdb)

Quando você só quer um resumo — uma contagem, uma média, um agrupamento —, dá
para consultar o arquivo diretamente com SQL. Nada é carregado na memória: o
duckdb lê apenas o necessário.

In [5]:
import duckdb

caminho_sql = str(arquivo).replace("\\", "/")

inicio = time.time()
resultado = duckdb.sql(f'''
    SELECT SG_UF_NOT AS uf,
           COUNT(*)  AS notificacoes
    FROM read_parquet('{caminho_sql}')
    GROUP BY SG_UF_NOT
    ORDER BY notificacoes DESC
    LIMIT 10
''').df()

print(f"Consulta em {time.time() - inicio:.2f}s, sem carregar o arquivo na memória")
resultado

Consulta em 0.07s, sem carregar o arquivo na memória


,uf,notificacoes
0,35,2182413
1,31,1658372
2,41,647663
3,42,336334
4,52,334182
5,33,302190
6,53,279020
7,29,231942
8,43,224791
9,32,137871


### Comparando as três

| Técnica | Tempo | Memória |
|---|---|---|
| Tabela inteira (`as_dataframe=True`) | ~23 s | ~29 GB |
| Só as colunas necessárias | ~1 s | ~1,4 GB |
| SQL com duckdb | menos de 1 s | praticamente nada |

Regra prática: **se você só quer um resumo, use SQL**. Se precisa manipular
linha a linha, leia as colunas necessárias.

## Consultas mais elaboradas

O SQL permite filtrar e agrupar em uma única passada:

In [6]:
consulta = duckdb.sql(f'''
    SELECT SUBSTRING(DT_NOTIFIC, 1, 7) AS mes,
           COUNT(*)                    AS notificacoes
    FROM read_parquet('{caminho_sql}')
    WHERE DT_NOTIFIC >= '2024-01-01'
      AND DT_NOTIFIC <= '2024-12-31'
    GROUP BY mes
    ORDER BY mes
''').df()

consulta

,mes,notificacoes
0,2024-01,339190
1,2024-02,996680
2,2024-03,1623904
3,2024-04,1700759
4,2024-05,1178021
5,2024-06,361635
6,2024-07,120308
7,2024-08,57082
8,2024-09,37800
9,2024-10,35542


## Vários arquivos de uma vez

O duckdb aceita uma lista de arquivos — útil para séries históricas.

In [7]:
caminhos_varios = []
for ano in (2022, 2023, 2024):
    c = sinan(disease="LEPT", year=ano, as_dataframe=False)
    caminhos_varios.append(str(c[0]).replace("\\", "/"))

lista_sql = ", ".join(f"'{c}'" for c in caminhos_varios)

serie = duckdb.sql(f'''
    SELECT NU_ANO   AS ano,
           COUNT(*) AS notificacoes
    FROM read_parquet([{lista_sql}])
    GROUP BY NU_ANO
    ORDER BY NU_ANO
''').df()

print("Leptospirose — notificações por ano:")
serie

Leptospirose — notificações por ano:


,ano,notificacoes
0,2022,15217
1,2023,20892
2,2024,27510


## Quando usar cada uma

- **`as_dataframe=True`** — bases estaduais (SIM, SINASC, CNES, SIH). São
  pequenas o bastante e o código fica mais simples.
- **Colunas selecionadas** — SINAN e SIA, quando você precisa dos dados linha a
  linha (filtrar, cruzar, exportar).
- **SQL com duckdb** — quando o resultado é um resumo: contagens, médias,
  agrupamentos, séries. É o mais rápido e o que menos consome memória.

---
*Notebook do projeto [PySusNoCode](https://github.com/cartaproale/PySusNoCode) —
um produto [Kraemer Academy](https://kraemeracademy.net).
Validado com dados reais do DATASUS.*